# CS336 Spring 2026 - 12 RoPE / Transformer Block / Transformer LM
----
> 09–11 已建立 Embedding、RMSNorm、SwiGLU、Causal Attention。本章将它们接成**具有 next-token logits 的完整小型 Transformer**。

```text
token IDs (B,T)
  ↓ Embedding
hidden (B,T,D)
  ↓ Transformer Blocks × L
  ↓ final RMSNorm
  ↓ LM head
logits (B,T,V)
```

模型输出是**未归一化的 logits**；训练时交给 Cross Entropy，生成时按温度与采样策略处理。


## 1. 为什么 Attention 需要位置信息？

11 中仅根据输入内容计算 Q、K、V，本身不包含绝对或相对位置信息。RoPE（Rotary Position Embedding）对每对相邻维度做按位置变化的二维旋转。

对一个二维向量：

$$
R(\theta)
\begin{pmatrix}x_0\\x_1\end{pmatrix}
=
\begin{pmatrix}
x_0\cos\theta-x_1\sin\theta\\
x_0\sin\theta+x_1\cos\theta
\end{pmatrix}
$$

其中不同维度对的角频率不同。RoPE 施加到 Q、K 上，而通常不施加到 V 上。


In [1]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(336)
torch.set_num_threads(2)


def apply_rope(x: torch.Tensor, theta_base: float = 10000.0):
    # x: (B,H,T,D_head), D_head 必须为偶数。
    b, h, t, d = x.shape
    if d % 2 != 0:
        raise ValueError("RoPE head dimension must be even")
    freq = theta_base ** (-torch.arange(0, d, 2, device=x.device).float() / d)
    positions = torch.arange(t, device=x.device).float()
    angles = positions[:, None] * freq[None, :]
    cos = angles.cos()[None, None, :, :].to(x.dtype)
    sin = angles.sin()[None, None, :, :].to(x.dtype)
    even = x[..., 0::2]
    odd = x[..., 1::2]
    return torch.stack(
        (even * cos - odd * sin, even * sin + odd * cos), dim=-1
    ).flatten(-2)


q = torch.randn(2, 3, 7, 8)
q_rotated = apply_rope(q)
assert q_rotated.shape == q.shape
torch.testing.assert_close(q[..., 0, :], q_rotated[..., 0, :])
print("RoPE:", tuple(q.shape), "→", tuple(q_rotated.shape))


RoPE: (2, 3, 7, 8) → (2, 3, 7, 8)


## 2. RoPE 的两条性质

旋转保持模长：

$$
\|R(\theta)x\|_2=\|x\|_2.
$$

两组向量若施加**相同位置**的旋转，它们的内积保持不变；若位置不同，Q、K 的内积会体现相对位置差异。

我们验证模长和“相同位置内积保持不变”。这是比“输出形状没变”更有信息量的数值性质。


In [2]:
k = torch.randn_like(q)
k_rotated = apply_rope(k)

torch.testing.assert_close(
    q.square().sum(-1), q_rotated.square().sum(-1), rtol=1e-5, atol=1e-5
)
torch.testing.assert_close(
    (q * k).sum(-1), (q_rotated * k_rotated).sum(-1), rtol=1e-5, atol=1e-5
)
try:
    apply_rope(torch.randn(1, 1, 3, 5))
    raise AssertionError("奇数 head_dim 必须拒绝")
except ValueError:
    pass
print("旋转模长 / 同位置内积 / 参数检查：通过")


旋转模长 / 同位置内积 / 参数检查：通过


## 3. 带 RoPE 的 Multi-Head Causal Self-Attention

与 11 相比，仅新增：

```text
q, k = project(x)
q, k = apply_rope(q), apply_rope(k)
attention(q, k, v)
```

仍然需要：

- `D % n_heads == 0`；
- `D_head % 2 == 0`；
- causal mask 严格屏蔽未来；
- 输出 projection 恢复 `(B,T,D)`。


In [3]:
class TeachingRoPEAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        if d_model % n_heads != 0 or (d_model // n_heads) % 2:
            raise ValueError("d_model / n_heads must be an even integer")
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.output_projection = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        b, t, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)

        def heads(z):
            return z.reshape(b, t, self.n_heads, self.head_dim).transpose(1, 2)

        q, k, v = heads(q), heads(k), heads(v)
        q, k = apply_rope(q), apply_rope(k)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        future = torch.ones(t, t, dtype=torch.bool, device=x.device).triu(1)
        scores = scores.masked_fill(future, float("-inf"))
        probabilities = torch.softmax(scores, dim=-1)
        y = probabilities @ v
        y = y.transpose(1, 2).contiguous().reshape(b, t, d)
        return self.output_projection(y)


attention = TeachingRoPEAttention(d_model=16, n_heads=4)
assert attention(torch.randn(2, 5, 16)).shape == (2, 5, 16)
print("RoPE MHA shape：通过")


RoPE MHA shape：通过


## 4. 构造 Pre-Norm Transformer Block

逐步恢复 10 的公式：

$$
h=x+\mathrm{Attention}(\mathrm{RMSNorm}_1(x))
$$

$$
y=h+\mathrm{SwiGLU}(\mathrm{RMSNorm}_2(h)).
$$

本教学版以 `nn.Linear` 承担投影，以便聚焦子层组合与数据流；09 已解释了它的矩阵乘法实现。


In [4]:
class TeachingRMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        xf = x.float()
        y = xf * torch.rsqrt(xf.square().mean(dim=-1, keepdim=True) + self.eps)
        return y.to(x.dtype) * self.weight.to(x.dtype)


class TeachingSwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class TeachingTransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.norm1 = TeachingRMSNorm(d_model)
        self.attn = TeachingRoPEAttention(d_model, n_heads)
        self.norm2 = TeachingRMSNorm(d_model)
        self.ffn = TeachingSwiGLU(d_model, d_ff)

    def forward(self, x):
        h = x + self.attn(self.norm1(x))
        return h + self.ffn(self.norm2(h))


block = TeachingTransformerBlock(16, 4, 32)
zz = torch.randn(2, 6, 16, requires_grad=True)
out = block(zz)
assert out.shape == zz.shape
out.square().mean().backward()
assert torch.isfinite(zz.grad).all()
print("Transformer Block forward + backward：通过")


Transformer Block forward + backward：通过


## 5. 组装 Transformer Language Model

先做 token embedding，然后叠加 `L` 个 Transformer Blocks，经过最终 RMSNorm，再投影到 vocabulary 维：

$$
\mathrm{logits}_{b,t,:}\in\mathbb R^V.
$$

我们不在 forward 里做 softmax，因为下一章的 Cross Entropy 应直接使用 logits，以获得更好的数值稳定性。


In [5]:
class TeachingTransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_context):
        super().__init__()
        self.max_context = max_context
        self.tok_embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList(
            [TeachingTransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        )
        self.final_norm = TeachingRMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, ids):
        if ids.ndim != 2:
            raise ValueError("input IDs must have shape (B,T)")
        if ids.shape[1] < 1 or ids.shape[1] > self.max_context:
            raise ValueError("invalid context length")
        x = self.tok_embedding(ids)
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.final_norm(x))


V, D, H, DFF, L = 32, 16, 4, 32, 2
model = TeachingTransformerLM(V, D, H, DFF, L, max_context=16)
ids = torch.randint(0, V, (2, 7))
logits = model(ids)
assert logits.shape == (2, 7, V)
print("input IDs:", tuple(ids.shape), "→ logits:", tuple(logits.shape))
print("trainable parameters:", sum(p.numel() for p in model.parameters()))


input IDs: (2, 7) → logits: (2, 7, 32)
trainable parameters: 6224


## 6. Causal LM 的 next-token objective

令输入 token 序列为：

```text
x0, x1, x2, x3, x4
```

模型在位置 `t` 的 logits 应预测位置 `t+1` 的真实 token。通过 shift：

```text
logits[:, :-1, :]  → 预测
input_ids[:, 1:]   → 标签
```

Cross Entropy 的 reduction 是所有有效 `(B,T-1)` token 位置的平均；注意区分 batch、time、vocab 三个轴。


In [6]:
tokens = torch.randint(0, V, (3, 8))
pred_logits = model(tokens)[:, :-1, :]
targets = tokens[:, 1:]
loss = F.cross_entropy(
    pred_logits.reshape(-1, V),
    targets.reshape(-1),
)
assert torch.isfinite(loss)
model.zero_grad(set_to_none=True)
loss.backward()
assert all(
    p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters()
)
print("shifted logits:", tuple(pred_logits.shape))
print("targets:", tuple(targets.shape))
print("cross entropy:", float(loss))
print("全模型 backward：通过")


shifted logits: (3, 7, 32)
targets: (3, 7)
cross entropy: 3.4554972648620605
全模型 backward：通过


/tmp/ipykernel_38987/1441690795.py:16: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("cross entropy:", float(loss))


## 7. Prefix invariance：未来 token 不能影响过去 logits

即使模型顺利 `.backward()`，仍可能把 causal mask 方向写反。本测试固定前缀，替换后面的 IDs，比较前缀位置的全部 vocabulary logits。

测试使用 `eval()`，确保未来有 dropout 时不会把随机性误判为泄漏。


In [7]:
model.eval()
a = torch.randint(0, V, (2, 8))
b = a.clone()
b[:, 5:] = (b[:, 5:] + 11) % V
with torch.no_grad():
    ya = model(a)
    yb = model(b)
torch.testing.assert_close(ya[:, :5], yb[:, :5], atol=1e-5, rtol=1e-5)
print("未来 token 改动不影响过去 logits：通过")


未来 token 改动不影响过去 logits：通过
